Misteral 7b with Qlora

In [ ]:
from pathlib import Path
import io, gzip, unicodedata
import os, random
import numpy as np
import regex as re
from xml.etree import ElementTree as ET

import torch
from datasets import Dataset, concatenate_datasets
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# basic config 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# paths
ROOT = Path(r"J:\FINAL PROJECT")
TMX_ES = ROOT / "data" / "en-es.tmx"
TMX_PT = ROOT / "data" / "en-pt.tmx"

# dataset sizes
MAX_SAMPLES = 2000
EVAL_SAMPLES = 500

# tower settings
TOWER_CKPT = "Unbabel/TowerInstruct-Mistral-7B-v0.2"  # model name on HF
MAX_LEN_TOWER = 212  # sequence length for training/eval

PROJECT_DIR = "./tower7b_scielo"
os.makedirs(PROJECT_DIR, exist_ok=True)


j:\FINAL PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
def _nfc_ws(s: str) -> str:
    s = unicodedata.normalize("NFC", s or "")
    s = s.replace("\u00A0", " ").replace("\u202F", " ")
    return re.sub(r"\s+", " ", s).strip()

def _norm_lang(tag: str) -> str:
    if not tag: return ""
    t = tag.replace("_", "-").lower()
    if t.startswith("eng"): return "en"
    if t.startswith("spa"): return "es"
    if t.startswith("por"): return "pt"
    return t.split("-")[0]

def _open_maybe_gz(path: str):
    return io.TextIOWrapper(gzip.open(path, "rb"), encoding="utf-8", errors="ignore") if path.endswith(".gz") \
           else io.open(path, "r", encoding="utf-8", errors="ignore")

def load_tmx_as_dataset(tmx_path: str, src="en", tgt="es", max_samples=None, seed=42) -> Dataset:
    p = Path(tmx_path); assert p.exists(), f"Missing TMX: {tmx_path}"
    src = _norm_lang(src); tgt = _norm_lang(tgt)
    pairs = []
    with _open_maybe_gz(str(p)) as fh:
        context = ET.iterparse(fh, events=("start", "end"))
        _, root = next(context)
        cur_src = cur_tgt = None
        for event, elem in context:
            tag = elem.tag.lower()
            if event == "start" and tag.endswith("tu"):
                cur_src = cur_tgt = None
            if event == "end" and tag.endswith("tuv"):
                lang = _norm_lang(elem.attrib.get("{http://www.w3.org/XML/1998/namespace}lang",
                                                  elem.attrib.get("lang","")))
                seg = next((c for c in elem if c.tag.lower().endswith("seg")), None)
                if seg is not None:
                    text = _nfc_ws("".join(seg.itertext()))
                    if lang == src: cur_src = text
                    elif lang == tgt: cur_tgt = text
            if event == "end" and tag.endswith("tu"):
                if cur_src and cur_tgt:
                    pairs.append((cur_src, cur_tgt))
                root.clear()
                if max_samples and len(pairs) >= max_samples:
                    break

    if max_samples and len(pairs) > max_samples:
        rnd = random.Random(seed)
        idx = list(range(len(pairs))); rnd.shuffle(idx); idx = idx[:max_samples]
        pairs = [pairs[i] for i in idx]

    return Dataset.from_dict({
        "src_text": [s for s, _ in pairs],
        "tgt_text": [t for _, t in pairs],
    })

def light_clean(ds: Dataset, src_lang_check="en"):
    import langid
    langid.set_languages([src_lang_check])

    URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
    PUNCT_ONLY_RE = re.compile(r"^\p{P}+$", re.UNICODE)

    def clean_example(ex):
        s = URL_RE.sub("<URL>", _nfc_ws(ex["src_text"]))
        t = URL_RE.sub("<URL>", _nfc_ws(ex["tgt_text"]))
        return {"src_text": s, "tgt_text": t}

    def ok(ex):
        s, t = ex["src_text"], ex["tgt_text"]
        if len(s.split()) < 2 or len(t.split()) < 2: return False
        if PUNCT_ONLY_RE.match(s) or PUNCT_ONLY_RE.match(t): return False
        if len(s.split()) > 200 or len(t.split()) > 200: return False
        ls, cs = langid.classify(s)
        if ls != src_lang_check and cs > 0.7: return False
        return True

    def dedup(ds_):
        seen, keep = set(), []
        for i, ex in enumerate(ds_):
            k = (ex["src_text"].lower(), ex["tgt_text"].lower())
            if k not in seen:
                seen.add(k); keep.append(i)
        return ds_.select(keep)

    ds = ds.map(clean_example)
    ds = dedup(ds)
    ds = ds.filter(ok)
    return ds


In [ ]:
# Load and clean
ds_es_all = light_clean(load_tmx_as_dataset(TMX_ES, "en", "es", MAX_SAMPLES + EVAL_SAMPLES, SEED))
ds_pt_all = light_clean(load_tmx_as_dataset(TMX_PT, "en", "pt", MAX_SAMPLES + EVAL_SAMPLES, SEED))

print("Total EN→ES pairs after cleaning:", len(ds_es_all))
print("Total EN→PT pairs after cleaning:", len(ds_pt_all))

es_eval = min(EVAL_SAMPLES, max(50, int(0.10 * len(ds_es_all))))
pt_eval = min(EVAL_SAMPLES, max(50, int(0.10 * len(ds_pt_all))))

split_es = ds_es_all.train_test_split(test_size=es_eval, seed=SEED, shuffle=True)
split_pt = ds_pt_all.train_test_split(test_size=pt_eval, seed=SEED, shuffle=True)

train_es, valid_es = split_es["train"], split_es["test"]
train_pt, valid_pt = split_pt["train"], split_pt["test"]

print("Final splits:")
print(f" EN→ES train {len(train_es)} / valid {len(valid_es)}")
print(f" EN→PT train {len(train_pt)} / valid {len(valid_pt)}")


Filter: 100%|██████████| 2499/2499 [00:00<00:00, 10895.77 examples/s]

Total EN→ES pairs after cleaning: 2497
Total EN→PT pairs after cleaning: 2488
Final splits:
 EN→ES train 2248 / valid 249
 EN→PT train 2240 / valid 248


In [ ]:
# Load Tower tokenizer
tower_tok = AutoTokenizer.from_pretrained(TOWER_CKPT, use_fast=True)

# Many chat models use eos as pad
if tower_tok.pad_token is None:
    tower_tok.pad_token = tower_tok.eos_token

# Add language fields
def add_lang_fields(ds, tgt_lang):
    return ds.map(lambda ex: {"src_lang": "en", "tgt_lang": tgt_lang})

train_es_instr = add_lang_fields(train_es, "es")
train_pt_instr = add_lang_fields(train_pt, "pt")
valid_es_instr = add_lang_fields(valid_es, "es")
valid_pt_instr = add_lang_fields(valid_pt, "pt")

# Concatenate EN to ES and EN to PT into one training / valid dataset
tower_train_raw = concatenate_datasets([train_es_instr, train_pt_instr])
tower_valid_raw = concatenate_datasets([valid_es_instr, valid_pt_instr])

print("Tower train size:", len(tower_train_raw))
print("Tower valid size:", len(tower_valid_raw))

def format_for_tower(ex):
    if ex["tgt_lang"] == "es":
        direction = "English to Spanish"
    else:
        direction = "English to Portuguese"

    user_prompt = (
        f"Translate the following scientific sentence from {direction}.\n"
        f"Source (en): {ex['src_text']}\n"
        f"Target ({ex['tgt_lang']}):"
    )
    assistant_answer = ex["tgt_text"]

    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_answer},
    ]

    text = tower_tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

tower_train_text = tower_train_raw.map(
    format_for_tower,
    remove_columns=tower_train_raw.column_names
)
tower_valid_text = tower_valid_raw.map(
    format_for_tower,
    remove_columns=tower_valid_raw.column_names
)

print("Sample formatted training example:\n")
print(tower_train_text[0]["text"][:600])


Map: 100%|██████████| 248/248 [00:00<00:00, 26911.61 examples/s]


Tower train size: 4488
Tower valid size: 497


Map: 100%|██████████| 497/497 [00:00<00:00, 15964.53 examples/s]

Sample formatted training example:

<|im_start|>user
Translate the following scientific sentence from English to Spanish.
Source (en): An exhaustive reading of these texts entailed deconstructing the information from a critical perspective and establishing the intertextual relations from which the theoretical tensions that are discussed in this article emerged.
Target (es):<|im_end|>
<|im_start|>assistant
La lectura exhaustiva del conjunto de textos, implicó deconstruir el saber desde una perspectiva crítica y establecer relaciones intertextuales donde emergieron las tensiones teóricas que se discuten en este artículo.<|im_end|>


In [ ]:
for i in range(3):
    print("---- Example", i, "----")
    print(tower_train_text[i]["text"])
    print()


---- Example 0 ----
<|im_start|>user
Translate the following scientific sentence from English to Spanish.
Source (en): An exhaustive reading of these texts entailed deconstructing the information from a critical perspective and establishing the intertextual relations from which the theoretical tensions that are discussed in this article emerged.
Target (es):<|im_end|>
<|im_start|>assistant
La lectura exhaustiva del conjunto de textos, implicó deconstruir el saber desde una perspectiva crítica y establecer relaciones intertextuales donde emergieron las tensiones teóricas que se discuten en este artículo.<|im_end|>


---- Example 1 ----
<|im_start|>user
Translate the following scientific sentence from English to Spanish.
Source (en): Mr. Newton does not give room to discussion, so what are we to do?
Target (es):<|im_end|>
<|im_start|>assistant
Pueden ser decretos y acuerdos al más alto nivel, así como modificaciones legislativas.<|im_end|>


---- Example 2 ----
<|im_start|>user
Translate

In [ ]:
def tokenize_tower(batch):
    return tower_tok(
        batch["text"],
        max_length=MAX_LEN_TOWER,
        truncation=True,
        padding=False,  # padding done in collator
    )

tower_train_tok = tower_train_text.map(
    tokenize_tower, batched=True, remove_columns=["text"]
)
tower_valid_tok = tower_valid_text.map(
    tokenize_tower, batched=True, remove_columns=["text"]
)

print(tower_train_tok[0].keys())
print("First input_ids length:", len(tower_train_tok[0]["input_ids"]))


Map: 100%|██████████| 497/497 [00:00<00:00, 17804.66 examples/s]

dict_keys(['input_ids', 'attention_mask'])
First input_ids length: 133


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tower_base = AutoModelForCausalLM.from_pretrained(
    TOWER_CKPT,
    quantization_config=bnb_config,
    device_map="auto",
)

tower_base = prepare_model_for_kbit_training(tower_base)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

tower_lora = get_peft_model(tower_base, lora_config)
tower_lora.print_trainable_parameters()


Loading checkpoint shards: 100%|██████████| 3/3 [00:16<00:00,  5.34s/it]


trainable params: 20,971,520 || all params: 7,262,720,000 || trainable%: 0.2888


In [ ]:
tower_collator = DataCollatorForLanguageModeling(
    tokenizer=tower_tok,
    mlm=False,
)

TOWER_OUT_DIR = os.path.join(PROJECT_DIR, "ckpt_tower_lora")

tower_args = TrainingArguments(
    output_dir=TOWER_OUT_DIR,
    per_device_train_batch_size=1,   # keep small for 7B + 4bit
    gradient_accumulation_steps=8,   # effective batch size 8
    learning_rate=2e-4,
    max_steps=1000,                  # you can adjust later
    logging_steps=50,
    save_strategy="no",
    eval_strategy="no",
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to="none",
)

tower_trainer = Trainer(
    model=tower_lora,
    args=tower_args,
    train_dataset=tower_train_tok,
    eval_dataset=None,
    data_collator=tower_collator,
    tokenizer=tower_tok,
)

print(" Training Tower 7B (QLoRA) on SciELO …")
tower_trainer.train()

os.makedirs(TOWER_OUT_DIR, exist_ok=True)
tower_lora.save_pretrained(TOWER_OUT_DIR)
print(" LoRA adapter saved to:", TOWER_OUT_DIR)

C:\Users\jeeva\AppData\Local\Temp\ipykernel_21668\2876369858.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  tower_trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 32000, 'pad_token_id': 2}.


🚀 Training Tower 7B (QLoRA) on SciELO …


j:\FINAL PROJECT\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1153: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,1.237100
100,1.148100
150,1.134600
200,1.146900
250,1.085700
300,1.125400
350,1.095800
400,1.112900
450,1.081400
500,1.083100


✅ LoRA adapter saved to: ./tower7b_scielo\ckpt_tower_lora


In [ ]:
print(next(tower_lora.parameters()).device)


cuda:0


In [ ]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
ter_metric = evaluate.load("ter")


def tower_translate_batch(model, tok, src_texts, tgt_lang, max_new_tokens=128, batch_size=4):
    preds = []
    model.eval()

    for i in range(0, len(src_texts), batch_size):
        batch_src = src_texts[i:i+batch_size]
        messages_batch = []

        for s in batch_src:
            if tgt_lang == "es":
                direction = "English to Spanish"
            else:
                direction = "English to Portuguese"

            user_prompt = (
                f"Translate the following scientific sentence from {direction}.\n"
                f"Source (en): {s}\n"
                f"Target ({tgt_lang}):"
            )

            messages = [{"role": "user", "content": user_prompt}]
            prompt_str = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            messages_batch.append(prompt_str)

        enc = tok(
            messages_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN_TOWER,
        ).to(model.device)

        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        decoded = tok.batch_decode(out, skip_special_tokens=False)

        for full in decoded:
            marker = "<|im_start|> assistant"
            if marker in full:
                full = full.split(marker, 1)[1]

            full = full.replace("<|im_end|>", "")
            full = full.replace("</s>", "").replace("<s>", "")

            preds.append(full.strip())

    return preds


def eval_tower_on_pair(model, tok, valid_ds, tgt_lang, name):
    src = [r["src_text"] for r in valid_ds]
    refs = [r["tgt_text"] for r in valid_ds]

    preds = tower_translate_batch(model, tok, src, tgt_lang=tgt_lang)
    refs_bleu = [[r] for r in refs]

    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]
    ter = ter_metric.compute(predictions=preds, references=refs_bleu)["score"]
    exact = float(np.mean([int(p == r) for p, r in zip(preds, refs)]) * 100.0)

    print(
        f"{name}: BLEU {bleu:.2f} | chrF {chrf:.2f} | TER {ter:.2f} | exact {exact:.2f}% on {len(preds)}"
    )
    return {"bleu": bleu, "chrf": chrf, "ter": ter, "exact": exact}


In [ ]:
print(" Tower 7B EN→ES (SciELO, LoRA):")
res_es = eval_tower_on_pair(
    tower_lora, tower_tok, valid_es,
    tgt_lang="es",
    name="Tower7B EN→ES (SciELO)"
)

print("\n Tower 7B EN→PT (SciELO, LoRA):")
res_pt = eval_tower_on_pair(
    tower_lora, tower_tok, valid_pt,
    tgt_lang="pt",
    name="Tower7B EN→PT (SciELO)"
)


🔎 Tower 7B EN→ES (SciELO, LoRA):
Tower7B EN→ES (SciELO): BLEU 35.56 | chrF 60.96 | TER 56.30 | exact 0.80% on 249

🔎 Tower 7B EN→PT (SciELO, LoRA):
Tower7B EN→PT (SciELO): BLEU 49.43 | chrF 74.03 | TER 38.53 | exact 4.03% on 248


In [ ]:
# Pick a few examples from each dev set
N = 5
sample_es = [valid_es[i]["src_text"] for i in range(N)]
refs_es   = [valid_es[i]["tgt_text"] for i in range(N)]

# Use the SAME function used in eval
raw_preds_es = tower_translate_batch(tower_lora, tower_tok, sample_es, tgt_lang="es", max_new_tokens=128)

for i, (src, ref, pred) in enumerate(zip(sample_es, refs_es, raw_preds_es)):
    print(f"\n=== EN→ES EXAMPLE {i} ===")
    print("SRC:", src)
    print("REF:", ref)
    print("PRED:", pred)



=== EN→ES EXAMPLE 0 ===
SRC: Authorized artisans have to pay taxes and duties on a monthly basis; so far, no new permits have been granted.
REF: Los titulares de licencias deben pagar impuestos y aranceles mensualmente, y por el momento no se entregan nuevas licencias.
PRED: Los artesanos autorizados deben pagar mensualmente impuestos y derechos, hasta el momento no se han concedido nuevas licencias.

=== EN→ES EXAMPLE 1 ===
SRC: A gastrostomy tube was used for feeding 3 patients, either patient and family after being informed about its metabolic effects during the end-of-life phase.
REF: Siete pacientes estuvieron sedados solo un día. En 3 pacientes se mantuvo la alimentación por sonda de gastrostomía, por solicitud del paciente o bien de su familia y allegados.
PRED: En 3 pacientes se utilizó un tubo de gastrostomía para la alimentación, tanto al paciente como a la familia, previa información sobre sus efectos metabólicos en fase de fin de vida.

=== EN→ES EXAMPLE 2 ===
SRC: A commo

In [ ]:

# HYBRID MT for Tower7B (TM + RBMT + terminology + QLoRA)
import re
from typing import Dict, Tuple, Optional, List
import numpy as np

# ---- 1) RBMT-style protection: equations, LaTeX, citations, percents, numbers ----
EQ_PATTERN   = re.compile(r"\b(?:log|ln|exp|sin|cos|tan)\s*\([^)]*\)")
LATEX_EQ     = re.compile(r"\$[^$]+\$")
CIT_PATTERN  = re.compile(r"[A-Z][a-z]+ et al\.\s*\(\d{4}\)")
PERC_PATTERN = re.compile(r"\d+(?:[.,]\d+)?\s*%")
NUM_PATTERN  = re.compile(r"\d+(?:[.,]\d+)?")

PROTECT_PATTERNS = [LATEX_EQ, EQ_PATTERN, CIT_PATTERN, PERC_PATTERN, NUM_PATTERN]
PH_ENT_RE = re.compile(r"§E(\d+)§")

def rbmt_protect_entities(text: str) -> Tuple[str, Dict[str, str]]:
    """
    Replace protected entities with §E0§, §E1§, ...
    so the model does not hallucinate or corrupt them.
    """
    spans = []
    for pat in PROTECT_PATTERNS:
        for m in pat.finditer(text):
            spans.append((m.start(), m.end(), m.group(0)))

    # sort by start, longer first
    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))

    non_overlap = []
    last_end = -1
    for s, e, val in spans:
        if s >= last_end:
            non_overlap.append((s, e, val))
            last_end = e

    out_parts, ph_map = [], {}
    cur, k = 0, 0
    for s, e, val in non_overlap:
        if cur < s:
            out_parts.append(text[cur:s])
        ph = f"§E{k}§"
        ph_map[ph] = val
        out_parts.append(ph)
        cur = e
        k += 1
    out_parts.append(text[cur:])
    return "".join(out_parts), ph_map

def rbmt_restore_entities(text: str, ph_map: Dict[str, str]) -> str:
    """
    Put original entities back where §E0§, §E1§, ... appear.
    """
    def repl(m):
        ph = m.group(0)
        return ph_map.get(ph, ph)
    return PH_ENT_RE.sub(repl, text)


#  Terminology placeholders: EN term -> §T0§ -> target term 
TERM_ES = {
    "systematic review": "revisión sistemática",
    "confidence interval": "intervalo de confianza",
    "polymerase chain reaction": "reacción en cadena de la polimerasa",
}
TERM_PT = {
    "systematic review": "revisão sistemática",
    "confidence interval": "intervalo de confiança",
    "polymerase chain reaction": "reação em cadeia da polimerase",
}

PH_TERM_RE = re.compile(r"§T(\d+)§")

def protect_terms(text: str, lang: str) -> Tuple[str, Dict[str, str]]:
    """
    Replace known EN terms in the SOURCE with §T0§, §T1§, ...
    Then after translation we replace them with fixed ES/PT equivalents.
    """
    lex = TERM_ES if lang == "es" else TERM_PT
    items = sorted(lex.items(), key=lambda kv: len(kv[0]), reverse=True)

    ph_map: Dict[str, str] = {}
    out = text
    k = 0
    for en_term, tgt_term in items:
        ph = f"§T{k}§"
        pattern = re.compile(re.escape(en_term), flags=re.IGNORECASE)
        if pattern.search(out):
            out = pattern.sub(ph, out)
            ph_map[ph] = tgt_term
            k += 1
    return out, ph_map

def restore_terms(text: str, ph_map: Dict[str, str]) -> str:
    """
    Replace §T0§, §T1§ ... with their fixed target-language terms.
    """
    def repl(m):
        ph = m.group(0)
        return ph_map.get(ph, ph)
    return PH_TERM_RE.sub(repl, text)


# Translation Memory (TM) from SciELO train sets 
try:
    from rapidfuzz import process, fuzz
    HAVE_RAPIDFUZZ = True
    print(" rapidfuzz available (TM lookup enabled for Tower)")
except ImportError:
    HAVE_RAPIDFUZZ = False
    print(" rapidfuzz not installed (TM lookup = exact match only for Tower)")

tm_es: List[Tuple[str, str]] = list(zip(train_es["src_text"], train_es["tgt_text"]))
tm_pt: List[Tuple[str, str]] = list(zip(train_pt["src_text"], train_pt["tgt_text"]))

src_es_list = [s for s, _ in tm_es]
src_pt_list = [s for s, _ in tm_pt]

def tm_lookup(text: str, lang: str, threshold: int = 95) -> Tuple[Optional[str], float]:
    """
    SMT-style sentence TM:
      - find closest EN source in SciELO train set
      - if similarity >= threshold, return its human translation.
    """
    tm = tm_es if lang == "es" else tm_pt
    src_list = src_es_list if lang == "es" else src_pt_list
    if not tm:
        return None, 0.0

    q = (text or "").strip()
    if not q:
        return None, 0.0

    if HAVE_RAPIDFUZZ:
        best_src, score, idx = process.extractOne(q, src_list, scorer=fuzz.token_sort_ratio)
        if score >= threshold:
            return tm[idx][1], float(score)
        return None, float(score)
    else:
        for s, t in tm:
            if s.strip() == q:
                return t, 100.0
        return None, 0.0


# Single-sentence Tower translate helper (reuses your batch code) 
@torch.inference_mode()
def tower_translate_one(model, tok, src_text: str, tgt_lang: str,
                        max_new_tokens: int = 128) -> str:
    """
    Use the SAME prompting path as tower_translate_batch,
    but for a single sentence.
    """
    preds = tower_translate_batch(
        model,
        tok,
        [src_text],
        tgt_lang=tgt_lang,
        max_new_tokens=max_new_tokens,
        batch_size=1,
    )
    return preds[0]


# Full hybrid pipeline: TM -> RBMT protections + terms -> Tower 
def hybrid_translate_tower(text: str, tgt_lang: str,
                           tm_threshold: int = 95) -> Tuple[str, Dict]:
    """
    Hybrid order:
      1) Translation Memory (SciELO TM) if similarity >= tm_threshold
      2) Else: RBMT protect entities + protect terms
               -> Tower7B QLoRA
               -> restore terms + entities

    tgt_lang: 'es' or 'pt'
    """
    assert tgt_lang in ("es", "pt")

    # TM lookup on original English sentence
    tm_out, tm_score = tm_lookup(text, tgt_lang, threshold=tm_threshold)
    if tm_out is not None:
        return tm_out, {"mode": "tm", "tm_score": tm_score}

    # RBMT protections and terminology placeholders on SOURCE
    protected, ent_map = rbmt_protect_entities(text)
    protected, term_map = protect_terms(protected, tgt_lang)

    # Tower7B QLoRA translation
    hyp = tower_translate_one(tower_lora, tower_tok, protected, tgt_lang=tgt_lang)

    # Restore terminology + entities on TARGET
    hyp = restore_terms(hyp, term_map)
    hyp = rbmt_restore_entities(hyp, ent_map)

    return hyp, {
        "mode": "tower",
        "tm_score": tm_score,
        "protected_entities": len(ent_map),
        "protected_terms": len(term_map),
    }


def translate_es_hybrid_tower(text: str, tm_threshold: int = 95) -> str:
    return hybrid_translate_tower(text, "es", tm_threshold=tm_threshold)[0]

def translate_pt_hybrid_tower(text: str, tm_threshold: int = 95) -> str:
    return hybrid_translate_tower(text, "pt", tm_threshold=tm_threshold)[0]


#  Evaluation: compare Hybrid Tower vs plain Tower 
# (re-use your BLEU/chrF metrics; if not defined, load them here)
try:
    bleu_metric
    chrf_metric
    ter_metric
except NameError:
    import evaluate
    bleu_metric = evaluate.load("sacrebleu")
    chrf_metric = evaluate.load("chrf")
    ter_metric = evaluate.load("ter")

def eval_hybrid_tower(valid_ds, tgt_lang: str, name: str,
                      tm_threshold: int = 95, limit: Optional[int] = None):
    src  = [r["src_text"] for r in valid_ds]
    refs = [r["tgt_text"] for r in valid_ds]
    if limit is not None:
        src, refs = src[:limit], refs[:limit]

    preds = []
    for s in src:
        hyp, meta = hybrid_translate_tower(s, tgt_lang, tm_threshold=tm_threshold)
        preds.append(hyp)

    refs_bleu = [[r] for r in refs]
    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]
    ter   = ter_metric.compute(predictions=preds, references=refs_bleu)["score"]
    exact = float(np.mean([p == r for p, r in zip(preds, refs)]) * 100.0)

    print(f"{name} (thr={tm_threshold}) on {len(preds)} → "
         f"BLEU {bleu:.2f} | chrF {chrf:.2f} | TER {ter:.2f} | Exact {exact:.2f}%")
    return {"bleu": bleu, "chrf": chrf, "exact": exact}


# Run hybrid eval + small smoke tests 
print("\n Hybrid Tower7B EN→ES:")
hy_es_tower = eval_hybrid_tower(
    valid_es,
    tgt_lang="es",
    name="Hybrid TM+RBMT+Tower7B EN→ES",
    tm_threshold=95,
    limit=128,   # you can increase later
)

print("\n Hybrid Tower7B EN→PT:")
hy_pt_tower = eval_hybrid_tower(
    valid_pt,
    tgt_lang="pt",
    name="Hybrid TM+RBMT+Tower7B EN→PT",
    tm_threshold=95,
    limit=128,
)

print("\nSMOKE TEST (Tower Hybrid ES):")
print(translate_es_hybrid_tower(
    "This systematic review reports a confidence interval of 95% and log(1+2)=sin(theta)."
))

print("\nSMOKE TEST (Tower Hybrid PT):")
print(translate_pt_hybrid_tower(
    "Hybrid machine translation model for scientific research papers (Smith et al. (2020)) with 12.5% improvement."
))


✅ rapidfuzz available (TM lookup enabled for Tower)

🔀 Hybrid Tower7B EN→ES:
Hybrid TM+RBMT+Tower7B EN→ES (thr=95) on 128 → BLEU 33.73 | chrF 60.13 | Exact 1.56%

🔀 Hybrid Tower7B EN→PT:
Hybrid TM+RBMT+Tower7B EN→PT (thr=95) on 128 → BLEU 48.77 | chrF 73.34 | Exact 2.34%

SMOKE TEST (Tower Hybrid ES):
Este revisión sistemática reporta un intervalo de confianza de un 95% y un log(1+2)=sin(theta).

SMOKE TEST (Tower Hybrid PT):
Modelo de tradução automática híbrida para trabalhos científicos (Smith et al. (2020)) com melhoria de 12.5%.
